# Model Candidate Comparison

Comparing five classifiers for AHU fault detection on the same train/val
split, using per-class precision, recall, and F1 (not plain accuracy, given
the severe class imbalance found in EDA). The goal is to pick a model with
evidence, not assumption.

**Candidates**:
- Logistic Regression - linear baseline, establishes a floor
- Random Forest - tree-based bagging ensemble
- XGBoost - gradient-boosted trees, handles missing values natively
- LightGBM - gradient-boosted trees, XGBoost's closest rival implementation
- MLP (feed-forward neural net) - tests whether deep learning helps on this
  tabular dataset at this scale, where research generally favors tree-based
  methods

**All 5 classes included**, including Valve position fault (71 train /
23 val rows). This class is extremely thin, so its metrics here should be
read as noisy and indicative rather than precise; SMOTE oversampling for
this specific class is applied later during final model training
(notebooks/03_model_training.ipynb), not in this candidate-comparison
stage, since SMOTE should only ever be evaluated in the context of the
model actually being shipped, not while still comparing candidates.

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv("../.env")

user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]
db = os.environ["POSTGRES_DB"]
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")

train_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'train'", engine)
val_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'val'", engine)

print(f"Train: {train_df.shape}")
print(f"Val: {val_df.shape}")
train_df["labeling"].value_counts()

Train: (105318, 23)
Val: (35107, 23)


labeling
Normal condition                71640
Supply fan fault                28789
Return air temperature fault     4199
Supply air temperature fault      619
Valve position fault               71
Name: count, dtype: int64

## Feature preparation

Separating the 18 sensor columns as features from metadata columns
(id, ahu_id, reading_time, split) and the target (labeling). The target is
label-encoded to integers. XGBoost and LightGBM receive true NaN for the
structural nulls found in EDA, since both handle missing values natively.
Logistic Regression, Random Forest, and MLP cannot handle NaN, so they
receive a copy with nulls filled as -1, a sentinel value distinguishable
from any real sensor reading, rather than an imputed average that would
misrepresent an inactive heating or cooling system as active.

In [2]:
from sklearn.preprocessing import LabelEncoder

feature_cols = [c for c in train_df.columns if c not in
                ("id", "ahu_id", "reading_time", "labeling", "split")]

X_train = train_df[feature_cols]
X_val = val_df[feature_cols]

le = LabelEncoder()
y_train = le.fit_transform(train_df["labeling"])
y_val = le.transform(val_df["labeling"])

print("Features:", feature_cols)
print("Classes:", list(le.classes_))

# Sentinel-filled version for models that can't handle NaN natively
X_train_filled = X_train.fillna(-1)
X_val_filled = X_val.fillna(-1)

Features: ['set_point_temp', 'return_temp', 'supply_air_temp', 'supply_fan', 'valve_position', 'heating_supply_temp_1', 'heating_supply_temp_2', 'total_heating_pump', 'heating_pump_1', 'heating_pump_2', 'heating_pump_3', 'cooling_supply_temp_1', 'cooling_supply_temp_2', 'total_cooling_pump', 'cooling_pump_1', 'cooling_pump_2', 'cooling_pump_3', 'cooling_pump_4']
Classes: ['Normal condition', 'Return air temperature fault', 'Supply air temperature fault', 'Supply fan fault', 'Valve position fault']


## Training all five candidates

Each model is trained on the same training set and evaluated on the same
validation set. Logistic Regression additionally needs feature scaling
(it is sensitive to feature magnitude, unlike tree-based models), so a
scaled copy is prepared for it and MLP, which also benefits from scaled
input.

In [3]:
import time
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
import xgboost as xgb
import lightgbm as lgb

# Logistic Regression and MLP benefit from scaled features;
# tree-based models (RF, XGBoost, LightGBM) do not need this.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_val_scaled = scaler.transform(X_val_filled)

results = {}


def train_and_time(name, fit_fn):
    print(f"Training {name}...")
    start = time.time()
    model = fit_fn()
    elapsed = time.time() - start
    print(f"  done in {elapsed:.1f}s")
    return model


# 1. Logistic Regression
lr = train_and_time("Logistic Regression",
    lambda: LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train))
results["Logistic Regression"] = lr.predict(X_val_scaled)

# 2. Random Forest
rf = train_and_time("Random Forest",
    lambda: RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train_filled, y_train))
results["Random Forest"] = rf.predict(X_val_filled)

# 3. XGBoost - receives true NaN, not the filled version
xgb_model = train_and_time("XGBoost",
    lambda: xgb.XGBClassifier(random_state=42, eval_metric="mlogloss").fit(X_train, y_train))
results["XGBoost"] = xgb_model.predict(X_val)

# 4. LightGBM - also handles true NaN natively
lgb_model = train_and_time("LightGBM",
    lambda: lgb.LGBMClassifier(random_state=42, verbose=-1).fit(X_train, y_train))
results["LightGBM"] = lgb_model.predict(X_val)

# 5. MLP - scaled input, same as Logistic Regression
mlp = train_and_time("MLP",
    lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42).fit(X_train_scaled, y_train))
results["MLP"] = mlp.predict(X_val_scaled)

print("\nAll five models trained")

Training Logistic Regression...
  done in 10.1s
Training Random Forest...
  done in 2.2s
Training XGBoost...
  done in 3.8s
Training LightGBM...
  done in 1.4s
Training MLP...
  done in 18.2s

All five models trained


## Comparing metrics

Printing full per-class precision, recall, and F1 for each model,
macro F1 as the primary summary number given our class imbalance.

In [4]:
from sklearn.metrics import f1_score

summary_rows = []

for name, preds in results.items():
    report = classification_report(y_val, preds, target_names=le.classes_, output_dict=True)
    macro_f1 = report["macro avg"]["f1-score"]
    weighted_f1 = report["weighted avg"]["f1-score"]
    summary_rows.append({"model": name, "macro_f1": macro_f1, "weighted_f1": weighted_f1})

    print(f"=== {name} ===")
    print(classification_report(y_val, preds, target_names=le.classes_))
    print()

summary_df = pd.DataFrame(summary_rows).sort_values("macro_f1", ascending=False)
summary_df

=== Logistic Regression ===
                              precision    recall  f1-score   support

            Normal condition       0.88      0.87      0.87     23881
Return air temperature fault       0.64      0.46      0.54      1400
Supply air temperature fault       0.55      0.03      0.06       206
            Supply fan fault       0.73      0.79      0.76      9597
        Valve position fault       0.00      0.00      0.00        23

                    accuracy                           0.83     35107
                   macro avg       0.56      0.43      0.44     35107
                weighted avg       0.82      0.83      0.82     35107


=== Random Forest ===
                              precision    recall  f1-score   support

            Normal condition       1.00      1.00      1.00     23881
Return air temperature fault       0.99      0.98      0.98      1400
Supply air temperature fault       0.98      0.92      0.95       206
            Supply fan fault       

/home/bganeshk/projects/agentic-bms/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/bganeshk/projects/agentic-bms/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/bganeshk/projects/agentic-bms/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

,model,macro_f1,weighted_f1
2,XGBoost,0.978984,0.997830
1,Random Forest,0.966986,0.997819
4,MLP,0.959299,0.993890
3,LightGBM,0.682943,0.966359
0,Logistic Regression,0.444259,0.822205


In [5]:
# Check 1: Do train and val share any rows (by id)?
overlap = set(train_df["id"]) & set(val_df["id"])
print(f"Overlapping row ids between train and val: {len(overlap)}")

# Check 2: Does any single feature perfectly separate the classes?
# Group by label and look at each feature's range - if a feature never
# overlaps between classes, it alone could explain perfect scores.
for col in feature_cols:
    ranges = train_df.groupby("labeling")[col].agg(["min", "max"])
    print(f"\n{col}")
    print(ranges)

Overlapping row ids between train and val: 0

set_point_temp
                               min   max
labeling                                
Normal condition              10.0  30.0
Return air temperature fault  18.0  40.0
Supply air temperature fault  10.0  34.0
Supply fan fault              18.0  32.0
Valve position fault          18.0  26.0

return_temp
                                min    max
labeling                                  
Normal condition               5.16  29.99
Return air temperature fault   3.67  31.71
Supply air temperature fault   5.81  29.05
Supply fan fault              -1.63  32.89
Valve position fault          18.43  27.21

supply_air_temp
                                min    max
labeling                                  
Normal condition               9.58  48.76
Return air temperature fault  10.32  48.90
Supply air temperature fault  10.87  30.61
Supply fan fault               8.49  45.95
Valve position fault          18.36  27.89

supply_fan
        

In [6]:
# Check 3: XGBoost feature importance - is it spread out or dominated
# by one feature?
importance = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance

total_heating_pump       0.516212
supply_fan               0.162593
total_cooling_pump       0.133251
return_temp              0.069102
set_point_temp           0.048098
heating_pump_1           0.027353
valve_position           0.020416
supply_air_temp          0.009144
heating_supply_temp_1    0.004192
heating_pump_2           0.002595
heating_pump_3           0.001990
heating_supply_temp_2    0.001879
cooling_supply_temp_2    0.001144
cooling_supply_temp_1    0.001091
cooling_pump_1           0.000265
cooling_pump_3           0.000265
cooling_pump_2           0.000234
cooling_pump_4           0.000174
dtype: float32

### Checking for leakage, single-feature separability, and feature importance

No overlapping row ids between train and val, confirming no leakage. Every
feature's value range overlaps across all five classes (no single sensor
perfectly separates any class on its own). Feature importance from
XGBoost is spread across several sensors (total_heating_pump, supply_fan,
total_cooling_pump, return_temp, and set_point_temp together account for
most of the model's decisions), not dominated by one column.

Valve position fault's valve_position range is locked at exactly 0.0,
matching its definition (valve stuck fully closed). All its heating-side
readings are NaN, meaning every occurrence of this fault in the dataset
happened while the AHU was in cooling mode.

## Conclusion

| Model | Macro F1 | Weighted F1 |
|---|---|---|
| XGBoost | 0.9790 | 0.9978 |
| Random Forest | 0.9670 | 0.9978 |
| MLP | 0.9593 | 0.9939 |
| LightGBM | 0.6829 | 0.9664 |
| Logistic Regression | 0.4443 | 0.8222 |

**Decision: XGBoost.** It handles class imbalance across all five classes
well by default, including strong performance on the two thinnest classes,
Supply air temperature fault (0.95 F1) and Valve position fault (0.96 F1),
without any tuning or oversampling applied yet.

LightGBM struggled badly on the same two thin classes without tuning,
0.47 F1 on Supply air temperature fault and 0.12 F1 on Valve position
fault, dragging its macro F1 down to 0.68 despite strong performance on
the common classes. Logistic Regression failed completely on Valve
position fault (0.00 across precision, recall, and F1, never predicted it
once), consistent with its broader inability to separate classes
non-linearly.

**Note on Valve position fault's numbers**: with only 23 validation
examples, these specific metrics carry real uncertainty, a handful of
predictions either way meaningfully shifts the score. SMOTE oversampling
for this class is applied in notebooks/03_model_training.ipynb, evaluated
there specifically for whether it improves this class's reliability
further on the model actually being shipped.